# Material-fitted 3D local meshes and nonuniform face partitions

An exact oblique-interface transmission problem verifies physical area weights, tensor diffusion, mixed data and one-sided RT reconstruction. The archived nine patches are not a convergence-rate study.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
from functools import partial
from threadpoolctl import threadpool_limits
from IPython.display import Image, display
from pymhm.tetrahedral import TetraMesh
from pymhm.darcy3d import TriangularSkeleton, solve_darcy_3d
from pymhm.planar_fitting import fit_planar_material, planar_face_partitions
from pymhm.reconstruction3d import reconstruct_darcy_moments_3d

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "examples"))
from planar3d_data import Planar3DData

In [ ]:
data = Planar3DData()
cube = TetraMesh.unit_cube()
macro = TetraMesh(cube.points, cube.cells[:1])
skeleton = TriangularSkeleton(macro, degree=1,
    face_partitions=planar_face_partitions(macro, data.material))
fine = fit_planar_material(macro, data.material).mesh
face = int(macro.boundary_faces[0])
with threadpool_limits(1):
    solution = solve_darcy_3d(macro, degree=4, local_meshes=(fine,), skeleton=skeleton,
        permeability=data.material, source=data.source, dirichlet=data.pressure,
        neumann={face: partial(data.normal_flux, normal=macro.normals[face])})
    recovered = reconstruct_darcy_moments_3d(solution, degree=1)
    errors = dict(pressure=solution.l2_error(data.pressure),
                  flux=solution.flux_l2_error(data.flux),
                  reconstructed=recovered.flux_l2_error(data.flux))
assert max(errors.values()) < 1e-10
errors

In [ ]:
directory = ROOT / "examples/results/planar3d"
report = json.loads((directory / "campaign.json").read_text())
assert len(report["rows"]) == 9
for row in report["rows"]:
    assert hashlib.sha256((directory / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
    assert max(row["pressure_l2"], row["raw_flux_l2"], row["reconstructed_flux_l2"]) < 2e-8
[(r["macro_subdivisions"], r["boundary"], r["pressure_l2"], r["raw_flux_l2"],
  r["reconstructed_flux_l2"]) for r in report["rows"]]

## Actual geometry and physical fields

The section is z=0.37. Black/white lines show original macroface intersections. Flat values retain fine-cell ownership; error color scales describe floating-point errors in these exactly represented fields.

In [ ]:
for name in ("geometry", "fields", "components"):
    display(Image(filename=str(ROOT / "docs/figures/planar3d" / f"{name}.png")))